# 01 - Load and parse the EU AI Act and the GDPR

**Goal of this notebook:** turn the raw HTML files from EUR-Lex into clean, small pieces of text called **chunks**, each labelled with where it came from (regulation, article number, language, link).

Why this is the first step of a RAG assistant: the assistant can only cite "GDPR Art. 17(1)" if, somewhere in the pipeline, a piece of text still *knows* that it is Article 17, paragraph 1. If we threw the whole law into one big string, that knowledge would be gone. So we keep the structure while parsing.

Run the cells from top to bottom. Nothing here needs an API key or internet.

## 1. Setup and the list of source files

Task 2 downloaded eight HTML files into `data/raw/` (2 laws x 2 languages x 2 kinds of file). `sources.json` describes each of them. Read the `role` column carefully, it explains the next section.

In [1]:
import json
import textwrap
from pathlib import Path

import pandas as pd

from assistant.models import Citation
from assistant.parse import parse_all, parse_regulation

# The notebook may be started from the project root or from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

sources = pd.DataFrame(json.loads((RAW / "sources.json").read_text(encoding="utf-8")))
sources[["file", "regulation", "language", "kind", "role", "celex", "version_date"]]

,file,regulation,language,kind,role,celex,version_date
0,ai_act_de.html,AI_ACT,de,consolidated,articles,02024R1689-20260727,2026-07-27
1,ai_act_en.html,AI_ACT,en,consolidated,articles,02024R1689-20260727,2026-07-27
2,ai_act_de_original.html,AI_ACT,de,original,recitals,32024R1689,2024-07-12
3,ai_act_en_original.html,AI_ACT,en,original,recitals,32024R1689,2024-07-12
4,gdpr_de.html,GDPR,de,consolidated,articles,02016R0679-20160504,2016-05-04
5,gdpr_en.html,GDPR,en,consolidated,articles,02016R0679-20160504,2016-05-04
6,gdpr_de_original.html,GDPR,de,original,recitals,32016R0679,2016-05-04
7,gdpr_en_original.html,GDPR,en,original,recitals,32016R0679,2016-05-04


### Consolidated text vs. original text (why there are two kinds of file)

* **Consolidated** files (`kind = consolidated`, `role = articles`) are the law *as it is in force today*, with all later amendments merged in. That is the text we want to answer questions from, so **articles and annexes come from here**.
* **Original** files (`kind = original`, `role = recitals`) are the text exactly as first published in the Official Journal. The consolidated versions do not contain the *recitals* (the numbered "Whereas ..." paragraphs before Article 1), so **recitals come only from the originals**.

We never parse articles from the original files, otherwise every article would appear twice (once amended, once not).

## 2. What the raw HTML looks like

This is the start of Article 1 of the GDPR (English, consolidated). It is not the usual EUR-Lex web page but a cleaner XML-like format. The classes tell us what each piece is: `title-article-norm` is "Article 1", `stitle-article-norm` is the article title, `norm` is a paragraph and `no-parag` is its number.

In [2]:
raw_html = (RAW / "gdpr_en.html").read_text(encoding="utf-8")
start = raw_html.index('id="art_1"') - len('<div class="eli-subdivision" ')
print(raw_html[start : start + 1100])
print("...")

<div class="eli-subdivision" id="art_1">
               <p class="title-article-norm" id="id-bf4faaac-92af-42e9-8122-7f0ce4dc29e0">Article 1</p>
               <div class="eli-title" id="art_1.tit_1">
                  <p class="stitle-article-norm">Subject-matter and objectives</p>
               </div>
               <div class="norm">
                  <span class="no-parag">1.  </span>
                  <div class="norm inline-element">This Regulation lays down rules relating to the protection of natural persons with regard to the processing of personal data and rules relating to the free movement of personal data.</div>
               </div>
               <div class="norm">
                  <span class="no-parag">2.  </span>
                  <div class="norm inline-element">This Regulation protects fundamental rights and freedoms of natural persons and in particular their right to the protection of personal data.</div>
               </div>
               <div class="norm">
   

Notice also the amendment markers such as `▼M1` (`<p class="modref">`) in the AI Act. They say "this paragraph was changed by amending act M1". They are noise for a reader, so the parser removes them.

## 3. Parse one file

`parse_regulation` takes the HTML text plus a few facts about the file, and returns a list of `Chunk` objects. Here we parse the English GDPR.

In [3]:
entry = next(
    s for s in json.loads((RAW / "sources.json").read_text(encoding="utf-8"))
    if s["regulation"] == "GDPR" and s["language"] == "en" and s["role"] == "articles"
)
chunks_gdpr_en = parse_regulation(
    raw_html,
    regulation=entry["regulation"],
    language=entry["language"],
    source_url=entry["url"],
    celex=entry["celex"],
    version_date=entry["version_date"],
)
print(len(chunks_gdpr_en), "chunks from", entry["file"])
chunks_gdpr_en[0]

414 chunks from gdpr_en.html


Chunk(id='GDPR-en-article-1-1', text='Article 1 – Subject-matter and objectives\n(1) This Regulation lays down rules relating to the protection of natural persons with regard to the processing of personal data and rules relating to the free movement of personal data.', regulation='GDPR', language='en', type='article', number='1', paragraph='1', title='Subject-matter and objectives', source_url='https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02016R0679-20160504#art_1', celex='02016R0679-20160504', version_date='2016-05-04')

A `Chunk` has the text and its metadata:

* `id` is unique, e.g. `GDPR-en-article-1-1` = regulation, language, type, article number, paragraph.
* `text` always starts with the heading ("Article 1 - Subject-matter and objectives") so the chunk makes sense even when it is read completely alone, which is exactly how a retriever will hand it to the language model later.
* `source_url` is the EUR-Lex link with an anchor (`#art_1`) that jumps to that article.

## 4. Parse everything and look at it as a table

`parse_all()` reads `sources.json` and applies the right role to each file. We put the result into a pandas DataFrame because tables are the easiest way to explore data.

In [4]:
chunks = parse_all(RAW)
df = pd.DataFrame([c.model_dump() for c in chunks])
df["chars"] = df["text"].str.len()
print(len(df), "chunks in total")
df.drop(columns=["source_url", "celex", "version_date"]).assign(
    text=df["text"].str.replace("\n", " | ").str.slice(0, 90) + "..."
).sample(8, random_state=3)

3044 chunks in total


,id,text,regulation,language,type,number,paragraph,title,chars
718,AI_ACT-de-annex-VIII-B.3,Anhang VIII – Bei der Registrierung des Hochri...,AI_ACT,de,annex,VIII,B.3,Bei der Registrierung des Hochrisiko-KI-System...,203
529,AI_ACT-de-article-80-4,Artikel 80 – Verfahren für den Umgang mit KI-S...,AI_ACT,de,article,80,4,"Verfahren für den Umgang mit KI-Systemen, die ...",593
828,AI_ACT-en-article-3-57,Article 3 – Definitions | For the purposes of ...,AI_ACT,en,article,3,57,Definitions,645
1501,AI_ACT-en-annex-XII,Annex XII – Transparency information referred ...,AI_ACT,en,annex,XII,NaN,Transparency information referred to in Articl...,309
1686,AI_ACT-de-recital-177,Erwägungsgrund 177 | Um Rechtssicherheit zu ge...,AI_ACT,de,recital,177,NaN,NaN,1274
2337,GDPR-en-article-10,Article 10 – Processing of personal data relat...,GDPR,en,article,10,NaN,Processing of personal data relating to crimin...,537
2462,GDPR-en-article-40-3,Article 40 – Codes of conduct | (3) In additio...,GDPR,en,article,40,3,Codes of conduct,788
2703,GDPR-de-recital-6,Erwägungsgrund 6 | Rasche technologische Entwi...,GDPR,de,recital,6,NaN,NaN,793


### How many chunks of each kind?

* **article** chunks: one per numbered paragraph of an article (an article without numbered paragraphs is one chunk; the definitions in Article 3 / Article 4 are one chunk per definition).
* **recital** chunks: one per recital.
* **annex** chunks: one per annex point (the AI Act has annexes, the GDPR has none).

German and English should be perfectly symmetric, a quick sanity check that the parser treats both languages the same.

In [5]:
counts = df.groupby(["regulation", "type", "language"]).size().unstack("language")
counts

language             de   en
regulation type             
AI_ACT     annex    110  110
           article  645  645
           recital  180  180
GDPR       article  414  414
           recital  173  173

In [6]:
# Same provisions in DE and EN share a "key"; every key should exist in both languages.
keys = df.assign(key=df.apply(lambda r: (r.regulation, r.type, r.number, r.paragraph), axis=1))
by_lang = keys.groupby("language")["key"].apply(set)
print("In DE but not EN:", len(by_lang["de"] - by_lang["en"]))
print("In EN but not DE:", len(by_lang["en"] - by_lang["de"]))

# Articles with a letter suffix were inserted by the 2026 amendment of the AI Act.
suffixed = sorted({n for n in df[df.type == "article"].number if not n.isdigit()})
print("Articles with letter suffixes:", suffixed)

In DE but not EN: 0
In EN but not DE: 0
Articles with letter suffixes: ['4a', '60a', '75a', '75b', '75c', '75d']


## 5. One article, German and English side by side

GDPR Article 17 ("Right to erasure"), paragraph 1. The two chunks have the same `key` (regulation, type, number, paragraph) but different `language`. Later, this shared key lets us avoid showing the same provision twice.

In [7]:
def one(language, article="17", paragraph="1", regulation="GDPR"):
    row = df[
        (df.regulation == regulation) & (df.type == "article") & (df.number == article)
        & (df.paragraph == paragraph) & (df.language == language)
    ].iloc[0]
    return textwrap.shorten(row.text.replace("\n", " "), 700, placeholder=" ...")

side = pd.DataFrame({"DE": [one("de")], "EN": [one("en")]})
side.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

,DE,EN
0,"Artikel 17 – Recht auf Löschung („Recht auf Vergessenwerden“) (1) Die betroffene Person hat das Recht, von dem Verantwortlichen zu verlangen, dass sie betreffende personenbezogene Daten unverzüglich gelöscht werden, und der Verantwortliche ist verpflichtet, personenbezogene Daten unverzüglich zu löschen, sofern einer der folgenden Gründe zutrifft: a) Die personenbezogenen Daten sind für die Zwecke, für die sie erhoben oder auf sonstige Weise verarbeitet wurden, nicht mehr notwendig. b) Die betroffene Person widerruft ihre Einwilligung, auf die sich die Verarbeitung gemäß Artikel 6 Absatz 1 Buchstabe a oder Artikel 9 Absatz 2 Buchstabe a stützte, und es fehlt an einer anderweitigen ...","Article 17 – Right to erasure (‘right to be forgotten’) (1) The data subject shall have the right to obtain from the controller the erasure of personal data concerning him or her without undue delay and the controller shall have the obligation to erase personal data without undue delay where one of the following grounds applies: (a) the personal data are no longer necessary in relation to the purposes for which they were collected or otherwise processed; (b) the data subject withdraws consent on which the processing is based according to point (a) of Article 6(1), or point (a) of Article 9(2), and where there is no other legal ground for the processing; (c) the data subject objects to ..."


## 6. Why the metadata matters: building a citation

Because every chunk carries `regulation`, `type`, `number`, `paragraph` and `source_url`, a citation is just a formatting step. No guessing, no language model involved.

In [8]:
c = next(c for c in chunks if c.id == "GDPR-en-article-17-1")
citation = Citation(
    regulation=c.regulation, type=c.type, number=c.number, paragraph=c.paragraph,
    title=c.title, source_url=c.source_url, language=c.language,
)
print(citation.label)
print(citation.source_url)

# An annex point and a recital work the same way:
for chunk_id in ["AI_ACT-en-annex-III-4", "AI_ACT-en-recital-26"]:
    x = next(c for c in chunks if c.id == chunk_id)
    fields = ["regulation", "type", "number", "paragraph", "title", "source_url", "language"]
    cit = Citation(**{k: getattr(x, k) for k in fields})
    print(cit.label, "->", cit.source_url)

GDPR Art. 17(1)
https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02016R0679-20160504#art_17
AI Act Annex III point 4 -> https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02024R1689-20260727#anx_III
AI Act Recital 26 -> https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689#rct_26


## 7. A look at annex and recital chunks

An annex point and a recital, as the assistant will later see them. The first line is always the self-explanatory heading.

In [9]:
for chunk_id in ["AI_ACT-en-annex-III-4", "AI_ACT-en-recital-26"]:
    x = next(c for c in chunks if c.id == chunk_id)
    print(textwrap.shorten(x.text.replace("\n", " | "), 500, placeholder=" ..."))
    print()

Annex III – High-risk AI systems referred to in Article 6(2), point 4 | Employment, workers’ management and access to self-employment: | (a) AI systems intended to be used for the recruitment or selection of natural persons, in particular to place targeted job advertisements, to analyse and filter job applications, and to evaluate candidates; | (b) AI systems intended to be used to make decisions affecting terms of work-related relationships, the promotion or termination of work-related ...

Recital 26 | In order to introduce a proportionate and effective set of binding rules for AI systems, a clearly defined risk-based approach should be followed. That approach should tailor the type and content of such rules to the intensity and scope of the risks that AI systems can generate. It is therefore necessary to prohibit certain unacceptable AI practices, to lay down requirements for high-risk AI systems and obligations for the relevant operators, and to lay down transparency ...



## 8. Are any chunks very long?

Chunks that are too long are harder to match precisely and use more of the language model's context. We keep them whole for now and compare chunking strategies in the next stage. Here are the five longest:

In [10]:
df.nlargest(5, "chars")[["id", "chars"]].reset_index(drop=True)

,id,chars
0,AI_ACT-de-article-5-1,6453
1,GDPR-de-article-70-1,6327
2,GDPR-en-article-70-1,5768
3,AI_ACT-en-article-5-1,5446
4,AI_ACT-de-recital-53,5216


In [11]:
print(df["chars"].describe().round(0).to_string())

count    3044.0
mean      707.0
std       676.0
min        74.0
25%       306.0
50%       474.0
75%       837.0
max      6453.0


## What you learned

* A **chunk** is a small piece of the law that keeps its identity (which article, which paragraph, which language, which link).
* We split along the law's own structure (article paragraph / recital / annex point) instead of cutting every N characters.
* Articles and annexes come from the **consolidated** text (current law), recitals from the **original** text (the consolidated files do not contain them).
* The result is saved to `data/chunks.jsonl` by `python -m assistant.parse`. The next stages (chunking comparison, embeddings, retrieval) all start from this file.

**Try it yourself:** change the article and paragraph in section 5 (for example AI Act article 5, paragraph 1) and see how the German and English chunks line up.